# VARIANT-1 Experiment on Google Colab A100

This notebook runs the complete VARIANT-1 experiment with all improvements:
- Exponential deadline pressure
- Zone capacity constraints  
- Heavy task emphasis
- Burst arrivals
- Large-scale training (200k steps, 16 envs)
- **🚀 Async training for 3-4x GPU speedup**
- **📊 Order generation optimization (2x faster init)**

**Runtime**: 
- With async training: **~1.5-2.5 hours** on A100 GPU ⚡
- Without async: ~4-6 hours (legacy)

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Navigate to project (adjust path as needed)
import os
os.chdir('/content/drive/MyDrive/LogitHMARL')
!pwd
!ls -la

In [ ]:
# 3. Check GPU availability
!nvidia-smi

In [ ]:
# 4. Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q stable-baselines3
!pip install -q gymnasium
!pip install -q pandas matplotlib seaborn
!pip install -q imageio

In [ ]:
# 5. Verify installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# 6. Set MODE to full for complete training
import os
os.environ['MODE'] = 'full'

# ========== 异步训练配置 ==========
# 根据你的GPU和CPU核心数调整这些参数

# 并行环境数量（根据CPU核心数调整）
# - Google Colab A100: 推荐 16-32
# - 本地测试: 推荐 4-8
os.environ['N_ENVS'] = '16'  # 🔧 在这里修改并行环境数量

# 其他异步训练参数（通常不需要改）
os.environ['ASYNC_BATCH_SIZE'] = '256'        # 训练批量
os.environ['ASYNC_BUFFER_SIZE'] = '50000'     # 经验缓冲区
os.environ['ASYNC_COLLECT_THREADS'] = '4'     # CPU收集线程数
os.environ['ASYNC_UPDATE_INTERVAL'] = '10'    # 模型更新间隔

print("MODE set to: full")
print("\n训练配置:")
print(f"  - 训练步数: 200,000")
print(f"  - 批量大小: 2,048")
print(f"  - 并行环境: {os.environ['N_ENVS']}")
print(f"  - 异步批量: {os.environ['ASYNC_BATCH_SIZE']}")
print(f"  - 收集线程: {os.environ['ASYNC_COLLECT_THREADS']}")
print(f"  - 评估步数: 300")

print("\n💡 提示:")
print("  如果GPU利用率低，增加 N_ENVS")
print("  如果内存不足，减少 N_ENVS")

In [ ]:
# 6.1. Check system resources to determine optimal N_ENVS
import os
import psutil

# 获取CPU核心数
cpu_count = os.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024**3)

print("📊 系统资源:")
print(f"  - CPU核心数: {cpu_count}")
print(f"  - 总内存: {ram_gb:.1f} GB")

# 推荐的N_ENVS
recommended_envs = min(cpu_count, 32)  # 最多32个
print(f"\n💡 推荐的 N_ENVS: {recommended_envs}")
print(f"   (基于 CPU 核心数，最多32)")

# 根据内存估算
mem_per_env_gb = 0.5  # 每个环境约0.5GB
max_envs_by_memory = int(ram_gb * 0.7 / mem_per_env_gb)  # 使用70%内存
print(f"   (基于内存，最多约 {max_envs_by_memory} 个)")

final_recommendation = min(recommended_envs, max_envs_by_memory)
print(f"\n✅ 最终推荐: N_ENVS = {final_recommendation}")

In [ ]:
# 6.5. Enable async training for 3-4x speedup (recommended for A100)
os.environ['USE_ASYNC'] = 'true'
print("\n🚀 Async training enabled!")
print("Benefits:")
print("  - GPU utilization: 15% → 60-75%")
print("  - Training speed: 3-4x faster")
print("  - CPU & GPU work in parallel")
print("\nTo disable async (use legacy serial training):")
print("  os.environ['USE_ASYNC'] = 'false'")

In [ ]:
# 7. Run the experiment (this will take 4-6 hours)
# Press Ctrl+C to interrupt if needed
!python run_experiments.py

## Alternative: Run in Background with nohup

In [ ]:
# Run in background (allows you to do other things)
%%bash
export MODE=full
nohup python run_experiments.py > variant1_colab.log 2>&1 &
echo "Experiment started in background. PID: $!"
echo "Check progress with: tail -f variant1_colab.log"

In [ ]:
# Monitor progress
!tail -100 variant1_colab.log

In [ ]:
# Check if still running
!ps aux | grep "python run_experiments.py" | grep -v grep

In [ ]:
# Monitor GPU usage
!nvidia-smi

## View Results

In [ ]:
# Load and display results
import pandas as pd

results = pd.read_csv('results/results.csv')
results_sorted = results.sort_values('total_value', ascending=False)

print("\n" + "="*80)
print("VARIANT-1 EXPERIMENT RESULTS")
print("="*80)
print(results_sorted.to_string(index=False))
print("\n")

# Highlight NL-HMARL vs S-Shape
nl_score = results[results['method'] == 'NL-HMARL']['total_value'].values[0]
ss_score = results[results['method'] == 'S-Shape']['total_value'].values[0]
print(f"NL-HMARL: {nl_score:,}")
print(f"S-Shape:  {ss_score:,}")
print(f"Difference: {nl_score - ss_score:,} ({((nl_score/ss_score - 1)*100):.1f}%)")

In [ ]:
# Visualize results
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.barh(results_sorted['method'], results_sorted['total_value'])
plt.xlabel('Total Value')
plt.title('VARIANT-1 Performance Comparison')
plt.tight_layout()
plt.savefig('results/performance_comparison.png', dpi=150)
plt.show()

## Download Results

In [ ]:
# Download results CSV
from google.colab import files
files.download('results/results.csv')

In [ ]:
# Package all results for download
!zip -r variant1_results.zip results/
files.download('variant1_results.zip')

## Troubleshooting

In [ ]:
# If you need to restart, first check for running processes
!ps aux | grep python

In [ ]:
# Kill a stuck process (replace PID with actual process ID)
# !kill -9 PID

In [ ]:
# Check disk space
!df -h

In [ ]:
# View Python error traceback if crash occurred
!tail -200 variant1_colab.log